# Atenção (fusão texto×série) no período do comitê agêntico — 16 ativos, 2024–2026

Objetivo: treinar e avaliar o modelo de atenção com **consulta numérica** (mesma família do residual C) sobre o mesmo painel de preços congelado, mesmo universo de 16 ativos, mesmas datas de decisão semanais e mesma execução do comitê agêntico Llama (decisão no fechamento, fill no fechamento seguinte, 10 bps, pesos iguais em 4 ativos, rebalance só na troca de cesta). Comparação final no **subperíodo de teste out-of-sample** contra as curvas salvas do agêntico, do momentum Top 4 e do comprar-e-manter, recortadas à mesma janela e **rebasadas para base 1** na primeira sessão dela.

Validade e limitações declaradas:
- O modelo é treinado nos primeiros 70% das sessões, validado nos 15% seguintes (com purga de rótulos: amostra só entra no bloco se o rótulo terminar dentro do próprio bloco) e decide apenas no bloco final; o agêntico não teve treino neste painel, então a comparação é restrita ao subperíodo comum.
- Curvas salvas recortadas e rebasadas ignoram posições e custos anteriores à janela: a carteira real seria contínua. É leitura **diagnóstica de subperíodo**, não protocolo idêntico ao do início do histórico.
- As manchetes de 2024–2026 vêm da captura RSS retrospectiva (`data/live_news/capture.jsonl`): `first_seen_at` é a própria captura de 2026, **não disponibilidade histórica verificável**. O braço de texto é contrafactual retrospectivo; a política de atraso é conservadora (um pregão inteiro após a publicação).
- MOMENT-1-small (revisão fixa) e MiniLM congelados; histórico já inspecionado; sem inferência hospedada; custos de API não entram nas curvas.
- Execução obrigatória no Google Colab com GPU; nenhuma etapa roda em CPU local.
- Dependências: o Colab usa Python 3.12, sem wheel para `numpy==1.25.2` exigida pelo `momentfm`; por isso ele é instalado com `--no-deps` sobre `numpy==1.26.4`.

## Insumos — zip `latent_fusion_colab.zip` na raiz do Google Drive

O zip (formato zip real) contém `pyproject.toml`, `src/`, `scripts/`, `config/`, o painel `prices_2024_2026.parquet`, as manchetes `capture.jsonl` e as curvas salvas auditadas. Gere localmente com:

```bash
zip -rq latent_fusion_colab.zip pyproject.toml src scripts config \
  data/llama_news_api/prices_2024_2026.parquet \
  data/live_news/capture.jsonl \
  outputs/llama_consensus_backtest/equity.csv \
  outputs/consensus_numeric_validation/equity_comparison.csv
```

Use uma sessão de Colab com GPU (T4 basta; treino e embeddings levam poucos minutos).

In [ ]:
import os
ZIP_PATH = '/content/drive/MyDrive/latent_fusion_colab.zip'
OUT_DIR = '/content/drive/MyDrive/latent_fusion_attention_vs_agentic'
USE_NEWS = True
SEEDS = (42, 43, 44)
WINDOW = 32
SEQ_LEN = 10
HORIZON = 5
COST_BPS = 10.0
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')

In [ ]:
try:
    import google.colab  # noqa: F401
except ImportError as exc:
    raise RuntimeError('Execute exclusivamente no Google Colab com GPU') from exc
from google.colab import drive
drive.mount('/content/drive')
%cd /content
!unzip -oq "{ZIP_PATH}" -d /content
%pip install -q "numpy==1.26.4" "transformers==4.33.3" "huggingface-hub==0.24.0" "sentence-transformers==2.3.1" "pyarrow==23.0.1"
%pip install -q --no-deps "momentfm==0.1.4"

In [ ]:
import sys
from pathlib import Path
_root = Path('/content')
while not ((_root / 'src').exists() and (_root / 'pyproject.toml').exists()) and _root != _root.parent:
    _root = _root.parent
sys.path.insert(0, str(_root))
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Ative uma GPU no Colab; nenhuma etapa roda em CPU')
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)
_root, torch.cuda.get_device_name(0), torch.__version__

In [ ]:
import hashlib
import importlib.metadata
import json
import numpy as np
import pandas as pd
from datetime import datetime, timezone
from src.fusion.portfolio_execution import execute_portfolio, validate_prices
from src.fusion.backtest_metrics import portfolio_metrics
from src.fusion.llama_advanced_agent import weekly_decision_dates

ROOT = Path('/content')
PRICES_SHA256 = 'ff20adeaaa27f25d83bfd3c880b7251d6734567cbc022b1be6e96347e55eb733'
def _sha(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as fh:
        for chunk in iter(lambda: fh.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()
prices = pd.read_parquet(ROOT / 'data/llama_news_api/prices_2024_2026.parquet')
prices.index = pd.DatetimeIndex(prices.index)
if _sha(ROOT / 'data/llama_news_api/prices_2024_2026.parquet') != PRICES_SHA256:
    raise ValueError('Painel de preços difere do congelado e auditado')
TICKERS = list(prices.columns)
validate_prices(prices, TICKERS)
agentic = pd.read_csv(ROOT / 'outputs/llama_consensus_backtest/equity.csv', parse_dates=['session']).set_index('session')
saved = pd.read_csv(ROOT / 'outputs/consensus_numeric_validation/equity_comparison.csv', parse_dates=['session']).set_index('session')
if saved.index.min() < prices.index.min() or saved.index.max() > prices.index.max():
    raise ValueError('Curvas salvas fora do painel de preços')
{'sessions': len(prices), 'tickers': len(TICKERS), 'primeira': prices.index[0].date(), 'última': prices.index[-1].date(),
 'agentic_sessions': len(agentic), 'weekly_decisions': len(weekly_decision_dates(prices, '2024-08-01', prices.index[-1]))}

In [ ]:
logret = np.log(prices).diff()
known_ret = prices.pct_change(fill_method=None)
past_vol = np.sqrt(252 * (logret ** 2).rolling(HORIZON, min_periods=HORIZON).mean())
forward = prices.shift(-(HORIZON + 1)).div(prices.shift(-1)).sub(1)
market = forward.mean(axis=1, skipna=False)
target_end = pd.Series(prices.index, index=prices.index).shift(-(HORIZON + 1))
n = len(prices)
all_dates = prices.index
min_j = WINDOW + SEQ_LEN
train_end = all_dates[int(n * 0.70) - 1]
val_end = all_dates[int(n * 0.85) - 1]
boundaries = {'train': train_end, 'val': val_end, 'test': all_dates[-1]}
rows = {'train': [], 'val': [], 'test': []}
for ti, ticker in enumerate(TICKERS):
    for j in range(min_j, n):
        date, end = all_dates[j], target_end.iloc[j]
        value = forward.iloc[j, ti] - market.iloc[j]
        if pd.isna(end) or not np.isfinite(value):
            continue
        split = 'train' if date <= train_end else 'val' if date <= val_end else 'test'
        if end <= boundaries[split]:
            rows[split].append((ti, j))
splits = {k: np.asarray(v, dtype=np.int32) for k, v in rows.items()}
if not max(target_end.iloc[[j for _, j in rows['train']]]) < min(all_dates[[j for _, j in rows['val']]]):
    raise ValueError('Rótulo de treino atravessa a primeira decisão da validação')
if not max(target_end.iloc[[j for _, j in rows['val']]]) < min(all_dates[[j for _, j in rows['test']]]):
    raise ValueError('Rótulo de validação atravessa a primeira decisão do teste')
pd.DataFrame({k: {'amostras': len(v), 'primeira_decisão': all_dates[v[:, 1]].min(),
                  'última_decisão': all_dates[v[:, 1]].max()} for k, v in splits.items()}).T

In [ ]:
from momentfm import MOMENTPipeline
moment = MOMENTPipeline.from_pretrained('AutonLab/MOMENT-1-small', model_kwargs={'task_name': 'embedding'},
                                       revision='411e288267f82cce86296dbe4d6c8bc533cc162f')
moment.init()
moment.to('cuda').eval()
required = sorted((ti, j) for ti in range(len(TICKERS))
                  for j in range(min_j - SEQ_LEN + 1, n))
emb = {}
with torch.inference_mode():
    for start in range(0, len(required), 32):
        batch = required[start:start + 32]
        x = np.zeros((len(batch), 1, 512), dtype=np.float32)
        mask = np.zeros((len(batch), 512), dtype=bool)
        for k, (ti, j) in enumerate(batch):
            x[k, 0, :WINDOW] = known_ret.iloc[j - WINDOW + 1:j + 1, ti].to_numpy(dtype=float)
            mask[k, :WINDOW] = True
        out = moment(x_enc=torch.from_numpy(x).to('cuda'), input_mask=torch.from_numpy(mask).to('cuda')).embeddings
        for k, key in enumerate(batch):
            emb[key] = out[k].detach().cpu().numpy().astype(np.float32)
del moment
torch.cuda.empty_cache()
D_TS = len(next(iter(emb.values())))
if len(emb) != len(required) or any(not np.isfinite(v).all() for v in emb.values()):
    raise ValueError('Embeddings MOMENT incompletos ou não finitos')
{'embeddings': len(emb), 'dimensão': D_TS, 'por_ativo_data': True}

In [ ]:
news = None
if USE_NEWS:
    news_rows = []
    with open(ROOT / 'data/live_news/capture.jsonl', encoding='utf-8') as stream:
        for line in stream:
            row = json.loads(line)
            if row.get('ticker') not in TICKERS or not isinstance(row.get('title'), str) or not row['title'].strip():
                continue
            ts = pd.Timestamp(row.get('published_at'))
            if ts is pd.NaT:
                continue
            if ts.tzinfo is not None:
                ts = ts.tz_convert('UTC').tz_localize(None)
            pos = prices.index.searchsorted(ts.normalize(), side='right') + 1
            if pos < n:
                news_rows.append((TICKERS.index(row['ticker']), str(prices.index[pos].date()), row['title']))
    news = pd.DataFrame(news_rows, columns=['ti', 'decision', 'title']).drop_duplicates(['ti', 'decision', 'title'])
    from sentence_transformers import SentenceTransformer
    encoder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device='cuda')
    encoder.eval()
    vectors = encoder.encode(news['title'].tolist(), batch_size=128, convert_to_numpy=True,
                             normalize_embeddings=True, show_progress_bar=False).astype(np.float32)
    del encoder
    torch.cuda.empty_cache()
    news['vector'] = list(vectors)
    if not np.isfinite(vectors).all():
        raise ValueError('Embedding MiniLM não finito')
{'manchetes_retrospectivas': 0 if news is None else len(news),
 'período': None if news is None else (news['decision'].min(), news['decision'].max()),
 'política': 'atraso de um pregão completo após a publicação; first_seen da captura 2026 não é disponibilidade histórica'}

In [ ]:
from torch.utils.data import TensorDataset
news_by_key = {}
if news is not None:
    for record in news.itertuples(index=False):
        news_by_key.setdefault((record.ti, record.decision), []).append(record.vector)
D_EVENT = 385
def build(split):
    mom, vol, ret, ev, mk, tgt = [], [], [], [], [], []
    for ti, j in splits[split]:
        tokens = []
        for k in range(j - HORIZON + 1, j + 1):
            key = str(all_dates[k].date())
            for vector in news_by_key.get((ti, key), []):
                age = (j - k) / HORIZON
                tokens.append(np.concatenate((vector, [np.clip(age, 0, 1)]), dtype=np.float32))
        tokens = tokens[-8:]
        events = np.zeros((8, D_EVENT), dtype=np.float32)
        mask = np.zeros(8, dtype=bool)
        if tokens:
            events[-len(tokens):] = np.stack(tokens)
            mask[-len(tokens):] = True
        mom.append(np.stack([emb[(ti, j - offset)] for offset in range(SEQ_LEN - 1, -1, -1)]))
        vol.append(float(past_vol.iloc[j, ti]))
        ret.append(float(known_ret.iloc[j, ti]))
        ev.append(events)
        mk.append(mask)
        tgt.append(float(forward.iloc[j, ti] - market.iloc[j]))
    tensors = (torch.from_numpy(np.stack(mom)), torch.tensor(vol, dtype=torch.float32),
               torch.tensor(ret, dtype=torch.float32), torch.from_numpy(np.stack(ev)),
               torch.from_numpy(np.stack(mk)), torch.tensor(tgt, dtype=torch.float32))
    if any(not torch.isfinite(t).all() for t in tensors):
        raise ValueError('Tensor não finito no split ' + split)
    return TensorDataset(*tensors)
datasets = {split: build(split) for split in splits}
if news is not None:
    for split in ('train', 'val'):
        if not bool(datasets[split].tensors[4].any()):
            raise ValueError('Captura RSS retrospectiva sem manchetes elegíveis no split ' + split +
                             '; o residual não é treinável. Rode com USE_NEWS=False.')
pd.Series({split: float(ds.tensors[4].any(1).float().mean()) for split, ds in datasets.items()}).rename('fração_com_manchete')

In [ ]:
import copy
from src.fusion.models import TSOnlyPredictor
from src.fusion.event_model import EventResidual
from src.fusion.fast_residual_experiment import _SavedNumericBaseline
from src.fusion.event_experiment import _train, _scores
device = torch.device('cuda')
def seed_all(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
test_frame = pd.DataFrame({'ti': splits['test'][:, 0], 'j': splits['test'][:, 1]})
test_frame['date'] = all_dates[test_frame['j'].to_numpy()]
test_frame['ticker'] = [TICKERS[ti] for ti in test_frame['ti']]
scores_by_seed = {}
for seed in SEEDS:
    seed_all(seed)
    baseline = _SavedNumericBaseline(TSOnlyPredictor(d_ts=D_TS)).to(device)
    _train(baseline, datasets['train'], datasets['val'], device, epochs=8, patience=2)
    numeric, _ = _scores(baseline, datasets['test'], device)
    if news is None:
        attention = numeric.copy()
    else:
        seed_all(seed)
        model = EventResidual(copy.deepcopy(baseline), event_dim=D_EVENT, max_residual=0.02).to(device)
        torch.nn.init.zeros_(model.residual[-1].weight)
        torch.nn.init.zeros_(model.residual[-1].bias)
        _train(model, datasets['train'], datasets['val'], device, residual=True, epochs=8, patience=2)
        attention, mask = _scores(model, datasets['test'], device, residual=True)
        frozen = all(torch.equal(a.detach().cpu(), b.detach().cpu()) for a, b in
                     zip(baseline.state_dict().values(), model.baseline.state_dict().values()))
        if not frozen or not np.array_equal(attention[~mask], numeric[~mask]):
            raise ValueError('Residual alterou o baseline ou não retornou ao numérico sem manchete')
        del model
        torch.cuda.empty_cache()
    scores_by_seed[seed] = {'numeric': numeric, 'attention': attention}
    del baseline
    torch.cuda.empty_cache()
{seed: {name: float(np.mean(vals ** 2)) for name, vals in scores.items()} for seed, scores in scores_by_seed.items()}

In [ ]:
def curve(scores):
    frame = test_frame.assign(score=scores).pivot(index='date', columns='ticker', values='score').reindex(columns=TICKERS)
    weekly = [d for d in weekly_decision_dates(prices, str(frame.index.min().date()), str(prices.index[-1].date())) if d in frame.index]
    if not weekly:
        raise ValueError('Nenhuma decisão semanal com previsão completa no teste')
    def actions(date, held, known):
        chosen = set(frame.loc[date].sort_values(ascending=False).index[:4])
        return {t: ('BUY' if t in chosen else 'SELL') for t in TICKERS}
    def weights(date, held, known):
        chosen = set(frame.loc[date].sort_values(ascending=False).index[:4])
        return np.array([0.25 if ticker in chosen else 0.0 for ticker in TICKERS])
    return execute_portfolio(prices, TICKERS, weekly, actions, cost_bps=COST_BPS,
                             rebalance='on_change', weight_fn=weights)['series']
curves = {f'{name}_{seed}': curve(scores[name]).net for seed in SEEDS for name in ('numeric', 'attention')}
model_equity = pd.DataFrame({name: pd.concat([curves[f'{name}_{seed}'] for seed in SEEDS], axis=1).mean(axis=1)
                              for name in ('numeric', 'attention')})
window = model_equity.index
saved_window = saved.loc[window[0]:window[-1]].reindex(window)
if saved_window.isna().any().any():
    raise ValueError('Curvas salvas não cobrem todas as sessões da janela de teste')
rebased = {col: saved_window[col] / saved_window[col].iloc[0] for col in saved_window.columns}
comparison = {}
series_map = {'atenção (média 3 sementes)': model_equity['attention'],
              'numérico (média 3 sementes)': model_equity['numeric'],
              'Consenso Llama (salvo, rebasado)': rebased['Consenso Llama'],
              'Momentum Top 4 (salvo, rebasado)': rebased['Momentum numérico Top 4'],
              'Comprar e manter (rebasado)': rebased['Comprar e manter']}
for name, series in series_map.items():
    comparison[name] = portfolio_metrics(series, rebased['Comprar e manter'])
comparison['atenção vs Consenso Llama'] = portfolio_metrics(model_equity['attention'], rebased['Consenso Llama'])
comparison['atenção vs numérico'] = portfolio_metrics(model_equity['attention'], model_equity['numeric'])
summary = pd.DataFrame(comparison).T[['total_return', 'sharpe', 'max_drawdown', 'alpha_annual', 'beta',
                                       'paired_mean_annual', 'paired_ci95_annual_low', 'paired_ci95_annual_high']]
summary.assign(subperíodo=f'{window[0].date()} a {window[-1].date()}', sessões=len(window))

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from src.backtest.visualization import BG, PANEL, GOLD, GREEN, CYAN, WHITE, PURPLE
out = Path(OUT_DIR)
out.mkdir(parents=True, exist_ok=True)
fig, ax = plt.subplots(figsize=(14, 7))
fig.patch.set_facecolor(BG)
ax.set_facecolor(PANEL)
styles = {'atenção (média 3 sementes)': (CYAN, '-'), 'numérico (média 3 sementes)': (PURPLE, '-'),
          'Consenso Llama (salvo, rebasado)': (GOLD, '-'), 'Momentum Top 4 (salvo, rebasado)': (WHITE, '-'),
          'Comprar e manter (rebasado)': (GREEN, '--')}
for name, (color, style) in styles.items():
    ax.plot(window, series_map[name].values, label=name, color=color, linewidth=2.2, linestyle=style)
ax.axhline(1, color=WHITE, alpha=0.2, linestyle='--')
ax.set_title('Atenção vs comitê agêntico · subperíodo de teste fora da amostra', fontsize=16, color='white')
ax.set_ylabel('Capital (base 1 na janela; após custos)', fontsize=14, color='white')
ax.tick_params(colors='white', labelsize=12)
ax.grid(alpha=0.12)
ax.legend(fontsize=14, labelcolor='white', facecolor=PANEL, edgecolor=WHITE)
fig.tight_layout()
fig.savefig(out / 'attention_vs_agentic.png', dpi=300, facecolor=BG, edgecolor='none')
plt.close(fig)
str(out / 'attention_vs_agentic.png')

In [ ]:
manifest = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'device': torch.cuda.get_device_name(0),
    'seeds': SEEDS, 'window': WINDOW, 'seq_len': SEQ_LEN, 'horizon': HORIZON, 'cost_bps': COST_BPS,
    'target': 'P[t+6]/P[t+1]-1 menos média igualmente ponderada dos 16 ativos',
    'splits': {'train_end': str(train_end.date()), 'val_end': str(val_end.date()),
               'amostras': {k: int(len(v)) for k, v in splits.items()}},
    'news': {'used': news is not None, 'count': 0 if news is None else len(news),
             'policy': 'atraso de um pregão após a publicação; retrospectivo, first_seen da captura 2026 não é disponibilidade histórica'},
    'models': {'numeric': 'TSOnlyPredictor sobre MOMENT-1-small congelado (embedding por ativo e data)',
               'attention': 'EventResidual C: consulta numérica, atenção sobre manchetes MiniLM, residual <= 2%'},
    'saved_curves_source': ['outputs/llama_consensus_backtest/equity.csv',
                            'outputs/consensus_numeric_validation/equity_comparison.csv'],
    'saved_curves_policy': 'recorte à janela de teste e rebase base 1; carteira contínua real teria posições e custos anteriores — comparação diagnóstica de subperíodo',
    'hashes': {rel: _sha(ROOT / rel) for rel in ('data/llama_news_api/prices_2024_2026.parquet',
                                                 'data/live_news/capture.jsonl',
                                                 'outputs/llama_consensus_backtest/equity.csv',
                                                 'outputs/consensus_numeric_validation/equity_comparison.csv')},
    'versions': {name: importlib.metadata.version(name) for name in ('torch', 'momentfm',
                                                                     'sentence-transformers', 'transformers', 'pandas')},
    'interpretation': 'Comparação restrita ao subperíodo de teste out-of-sample; histórico já inspecionado; '
                      'manchetes retrospectivas não point-in-time; nada estabelece superioridade geral de LLM '
                      'nem de atenção; sem correção por seleção da pesquisa histórica.',
}
(out / 'run_manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
summary.to_csv(out / 'comparison_metrics.csv')
model_equity.to_csv(out / 'model_equity.csv')
pd.DataFrame(curves).to_csv(out / 'curves_by_seed.csv')
(out / 'results.md').write_text(
    '# Atenção vs comitê agêntico — subperíodo de teste\n\n'
    f'Janela: {window[0].date()} a {window[-1].date()} ({len(window)} sessões). Mesmas datas semanais de decisão, '
    'execução no fechamento seguinte, 10 bps, pesos iguais em 4 ativos com rebalance só na troca. Consenso Llama, '
    'Top 4 e comprar-e-manter vêm das curvas auditadas salvas, recortadas à mesma janela e rebasadas para base 1 '
    'na primeira sessão (carteira contínua real teria posições e custos anteriores: leitura diagnóstica).\n\n'
    '```\n' + summary.to_string() + '\n```\n\n'
    'Manchetes retrospectivas (first_seen da captura 2026), política de atraso de um pregão após a publicação; '
    'modelo treinado em 70% iniciais com purga, validação 15%, decisões apenas no bloco final. Histórico já '
    'inspecionado: leitura exploratória e diagnóstica, não confirmação independente. Ver run_manifest.json.\n',
    encoding='utf-8')
{'manifest': str(out / 'run_manifest.json'), 'resultados': str(out / 'results.md'), 'figura': str(out / 'attention_vs_agentic.png')}